<a href="https://colab.research.google.com/github/PawanKumar1216/northstar-databases-analytics-assignment/blob/main/notebooks/04_SQL_Analysis_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 SQL Analysis in R

## NorthStar Urban Mobility and Logistics

This notebook applies SQL queries within the R environment to analyse structured NorthStar operational data. It demonstrates CRUD operations, filtering, joins, aggregation, business interpretation, and an optimisation example using connected datasets loaded directly from the project GitHub repository.

In [2]:
# Install and load required packages
if (!require("sqldf")) install.packages("sqldf", repos = "https://cloud.r-project.org")

library(sqldf)

# GitHub raw data URL
base_url <- "https://raw.githubusercontent.com/PawanKumar1216/northstar-databases-analytics-assignment/main/data/"

# Load NorthStar dataset files directly from GitHub
customers <- read.csv(paste0(base_url, "customers.csv"))
deliveries <- read.csv(paste0(base_url, "deliveries.csv"))
orders <- read.csv(paste0(base_url, "orders.csv"))
complaints <- read.csv(paste0(base_url, "complaints.csv"))
drivers <- read.csv(paste0(base_url, "drivers.csv"))
hubs <- read.csv(paste0(base_url, "hubs.csv"))
incidents <- read.csv(paste0(base_url, "incidents.csv"))
vehicles <- read.csv(paste0(base_url, "vehicles.csv"))

print("All structured NorthStar dataset files loaded successfully from GitHub.")

[1] "All structured NorthStar dataset files loaded successfully from GitHub."


## 2. Initial Dataset Check

Before running SQL queries, the main structured datasets are checked to confirm that the files were loaded successfully and contain the expected number of records.


In [3]:
# Check dataset dimensions after import
dataset_dimensions <- data.frame(
  Dataset = c("Customers", "Deliveries", "Orders", "Complaints", "Drivers", "Hubs", "Incidents", "Vehicles"),
  Records = c(
    nrow(customers),
    nrow(deliveries),
    nrow(orders),
    nrow(complaints),
    nrow(drivers),
    nrow(hubs),
    nrow(incidents),
    nrow(vehicles)
  ),
  Columns = c(
    ncol(customers),
    ncol(deliveries),
    ncol(orders),
    ncol(complaints),
    ncol(drivers),
    ncol(hubs),
    ncol(incidents),
    ncol(vehicles)
  )
)

dataset_dimensions

Dataset,Records,Columns
<chr>,<int>,<int>
Customers,650,9
Deliveries,950,13
Orders,1250,11
Complaints,320,10
Drivers,170,8
Hubs,8,5
Incidents,280,7
Vehicles,120,8


## 3. CRUD Operations in R

CRUD operations are used to demonstrate basic relational data management within the R environment. A temporary customer record is created, read, updated, and deleted to show how operational records can be managed before deeper SQL analysis is performed.

### 3.1 Create Operation

A new temporary customer record is inserted into the `customers` dataset to demonstrate how new operational records can be added within the relational environment.

In [6]:
# Remove the temporary customer first if the cell is run more than once
customers <- customers[customers$customer_id != "C9999", ]

# Create a new temporary customer record
new_customer <- data.frame(
  customer_id = "C9999",
  age = 35,
  home_zone = "Central",
  customer_type = "Consumer",
  signup_date = "2026-05-10",
  loyalty_score = 80,
  app_engagement_score = 75,
  preferred_channel = "App",
  account_status = "Active"
)

# Add the new customer record to the customers dataset
customers <- rbind(customers, new_customer)

# Display the newly inserted customer record
sqldf("
  SELECT *
  FROM customers
  WHERE customer_id = 'C9999'
")

customer_id,age,home_zone,customer_type,signup_date,loyalty_score,app_engagement_score,preferred_channel,account_status
<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>
C9999,35,Central,Consumer,2026-05-10,80,75,App,Active


### 3.2 Read Operation

A SQL `SELECT` query is used to retrieve selected customer records from the `customers` dataset. This demonstrates relational data retrieval using SQL within the R environment.

In [7]:
# Read selected customer records using SQL in R
sqldf("
  SELECT customer_id, customer_type, loyalty_score, preferred_channel, account_status
  FROM customers
  WHERE customer_id IN ('C0001', 'C0002', 'C0003', 'C9999')
")

customer_id,customer_type,loyalty_score,preferred_channel,account_status
<chr>,<chr>,<dbl>,<chr>,<chr>
C0001,SME,44.9,App,Active
C0002,Consumer,55.4,App,Active
C0003,Consumer,75.9,,Active
C9999,Consumer,80.0,App,Active


### 3.3 Update Operation

The temporary customer record is updated by changing the account status from `Active` to `Inactive`. A SQL query is then used to verify that the update was applied successfully.

In [8]:
# Update the temporary customer record
customers$account_status[customers$customer_id == "C9999"] <- "Inactive"

# Verify the update using SQL in R
sqldf("
  SELECT customer_id, customer_type, account_status
  FROM customers
  WHERE customer_id = 'C9999'
")

customer_id,customer_type,account_status
<chr>,<chr>,<chr>
C9999,Consumer,Inactive


### 3.4 Delete Operation

The temporary customer record is deleted from the dataset. A verification query is then executed to confirm that the record has been removed successfully.

In [10]:
# Delete the temporary customer record
customers <- customers[customers$customer_id != "C9999", ]

# Verify deletion using SQL in R
sqldf("
  SELECT *
  FROM customers
  WHERE customer_id = 'C9999'
")

customer_id,age,home_zone,customer_type,signup_date,loyalty_score,app_engagement_score,preferred_channel,account_status
<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>


## 4. Analytical SQL Queries

The following SQL queries are executed within the R environment to analyse delivery performance, hub efficiency, driver performance, route behaviour, operational costs, complaints, incidents, and customer segments. These queries connect multiple NorthStar datasets and help identify the underlying operational problems described in the case study.

### 4.1 Query 1: Delivery Status Distribution

A SQL aggregation query is used to calculate the number of deliveries recorded as on time, delayed, or failed. This provides a baseline measure of service reliability across the NorthStar delivery operation.

In [11]:
# Query 1: Delivery status distribution
delivery_status_distribution <- sqldf("
  SELECT
    delivery_status,
    COUNT(*) AS total_deliveries,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM deliveries), 2) AS percentage
  FROM deliveries
  GROUP BY delivery_status
  ORDER BY total_deliveries DESC
")

delivery_status_distribution

delivery_status,total_deliveries,percentage
<chr>,<int>,<dbl>
OnTime,616,64.84
Delayed,202,21.26
Failed,132,13.89


### 4.2 Query 2: Hub Performance Analysis

A SQL query is used to compare delivery performance across operational hubs. Instead of treating every non-matching record as delayed, this query correctly separates on-time, delayed, and failed deliveries and calculates the percentage of deliveries that were not completed on time at each hub.

In [12]:
# Query 2: Hub performance analysis
hub_performance <- sqldf("
  SELECT
    h.hub_id,
    h.hub_name,
    h.zone,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'OnTime' THEN 1 ELSE 0 END) AS on_time_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
    ROUND(
      SUM(CASE WHEN d.delivery_status IN ('Delayed', 'Failed') THEN 1 ELSE 0 END) * 100.0
      / COUNT(d.delivery_id),
      2
    ) AS non_on_time_percentage
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY h.hub_id, h.hub_name, h.zone
  ORDER BY non_on_time_percentage DESC
")

hub_performance

hub_id,hub_name,zone,total_deliveries,on_time_deliveries,delayed_deliveries,failed_deliveries,non_on_time_percentage
<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<dbl>
H05,Central Core,Central,115,67,25,23,41.74
H06,Airport Hub,Airport,104,62,27,15,40.38
H08,Midtown Relay,Central,128,80,22,26,37.50
H04,West Gate,West,127,83,28,16,34.65
H02,South Link,South,106,70,26,10,33.96
H07,Riverside Hub,Riverside,115,76,25,14,33.91
H01,North Exchange,North,136,93,26,17,31.62
H03,East Dock,East,119,85,23,11,28.57


### 4.3 Query 3: Driver Performance Evaluation

A SQL query is used to evaluate driver performance by combining delivery records with driver information. The query calculates the number of deliveries handled, average customer rating, average route overrides, and average operational cost for each driver. This helps identify drivers who may require further performance review or targeted support.

In [13]:
# Query 3: Driver performance evaluation
driver_performance <- sqldf("
  SELECT
    dr.driver_id,
    dr.employment_type,
    dr.years_experience,
    COUNT(d.delivery_id) AS total_deliveries,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS average_customer_rating,
    ROUND(AVG(d.manual_route_override_count), 2) AS average_route_overrides,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS average_operational_cost
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  WHERE d.customer_rating_post_delivery IS NOT NULL
  GROUP BY dr.driver_id, dr.employment_type, dr.years_experience
  HAVING COUNT(d.delivery_id) >= 5
  ORDER BY average_customer_rating ASC, average_route_overrides DESC
  LIMIT 10
")

driver_performance

driver_id,employment_type,years_experience,total_deliveries,average_customer_rating,average_route_overrides,average_operational_cost
<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>
D141,PartTime,14,9,2.93,0.78,10.97
D165,PartTime,10,6,2.98,1.33,11.04
D091,FullTime,12,6,3.13,0.17,10.96
D053,FullTime,5,7,3.14,0.86,12.29
D002,FullTime,4,7,3.21,1.00,11.95
D095,FullTime,12,5,3.26,1.00,13.87
D016,FullTime,6,6,3.33,1.17,14.49
D011,FullTime,3,6,3.34,1.00,12.99
D100,FullTime,4,8,3.35,1.25,16.27


### 4.4 Query 4: Route Override Impact Analysis

A SQL query is used to investigate whether higher numbers of manual route overrides are associated with longer delivery distances, higher operational costs, and weaker delivery outcomes. Deliveries are grouped into no-override, low-override, and high-override categories.

In [14]:
# Query 4: Route override impact analysis
route_override_impact <- sqldf("
  SELECT
    CASE
      WHEN manual_route_override_count = 0 THEN 'No Override'
      WHEN manual_route_override_count BETWEEN 1 AND 2 THEN 'Low Override'
      ELSE 'High Override'
    END AS override_category,
    COUNT(*) AS total_deliveries,
    ROUND(AVG(route_distance_km), 2) AS average_route_distance_km,
    ROUND(AVG(fuel_or_charge_cost), 2) AS average_operational_cost,
    ROUND(AVG(customer_rating_post_delivery), 2) AS average_customer_rating,
    ROUND(
      SUM(CASE WHEN delivery_status IN ('Delayed', 'Failed') THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
      2
    ) AS non_on_time_percentage
  FROM deliveries
  GROUP BY override_category
  ORDER BY
    CASE override_category
      WHEN 'No Override' THEN 1
      WHEN 'Low Override' THEN 2
      WHEN 'High Override' THEN 3
    END
")

route_override_impact

override_category,total_deliveries,average_route_distance_km,average_operational_cost,average_customer_rating,non_on_time_percentage
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
No Override,399,12.65,12.43,3.90,31.08
Low Override,463,14.21,12.98,3.86,37.58
High Override,88,18.04,14.00,3.70,40.91


### 4.5 Query 5: Cost Efficiency Across Hubs

A SQL aggregation query is used to compare average operational costs across hubs. This helps identify locations where delivery operations may be more expensive and may require further cost-efficiency review.

In [15]:
# Query 5: Cost efficiency across hubs
hub_cost_efficiency <- sqldf("
  SELECT
    h.hub_id,
    h.hub_name,
    h.zone,
    COUNT(d.delivery_id) AS total_deliveries,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS average_operational_cost,
    ROUND(AVG(d.route_distance_km), 2) AS average_route_distance_km,
    ROUND(
      SUM(CASE WHEN d.delivery_status IN ('Delayed', 'Failed') THEN 1 ELSE 0 END) * 100.0
      / COUNT(d.delivery_id),
      2
    ) AS non_on_time_percentage
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY h.hub_id, h.hub_name, h.zone
  ORDER BY average_operational_cost DESC
")

hub_cost_efficiency

hub_id,hub_name,zone,total_deliveries,average_operational_cost,average_route_distance_km,non_on_time_percentage
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>
H05,Central Core,Central,115,13.69,14.32,41.74
H06,Airport Hub,Airport,104,13.32,14.41,40.38
H04,West Gate,West,127,13.17,13.38,34.65
H07,Riverside Hub,Riverside,115,12.92,14.29,33.91
H01,North Exchange,North,136,12.76,13.64,31.62
H03,East Dock,East,119,12.74,14.52,28.57
H02,South Link,South,106,12.57,14.17,33.96
H08,Midtown Relay,Central,128,11.71,12.82,37.50


### 4.6 Query 6: Delivery Outcomes and Complaints

A SQL join query is used to connect orders, deliveries, and complaints so that complaint behaviour can be analysed according to delivery outcome. This helps investigate whether delayed or failed deliveries are associated with higher customer complaint volumes.

In [17]:
# Query 6: Delivery outcomes and complaint behaviour
delivery_complaints <- sqldf("
  SELECT
    d.delivery_status,
    COUNT(DISTINCT d.delivery_id) AS total_deliveries,
    COUNT(DISTINCT CASE WHEN c.complaint_id IS NOT NULL THEN d.delivery_id END) AS deliveries_with_complaints,
    ROUND(
      COUNT(DISTINCT CASE WHEN c.complaint_id IS NOT NULL THEN d.delivery_id END) * 100.0
      / COUNT(DISTINCT d.delivery_id),
      2
    ) AS complaint_rate_percentage,
    COUNT(c.complaint_id) AS total_complaints,
    ROUND(AVG(c.compensation_amount), 2) AS average_compensation,
    ROUND(AVG(c.resolution_days), 2) AS average_resolution_days
  FROM deliveries d
  LEFT JOIN complaints c ON d.order_id = c.order_id
  GROUP BY d.delivery_status
  ORDER BY complaint_rate_percentage DESC
")

delivery_complaints

delivery_status,total_deliveries,deliveries_with_complaints,complaint_rate_percentage,total_complaints,average_compensation,average_resolution_days
<chr>,<int>,<int>,<dbl>,<int>,<dbl>,<dbl>
Failed,132,33,25.00,35,25.47,9.66
Delayed,202,45,22.28,48,18.36,7.63
OnTime,616,131,21.27,149,19.18,7.60


### 4.7 Query 7: Incident Frequency and Severity Analysis

A SQL query is used to analyse the most common operational incidents, their severity, and the average time required to resolve them. This helps identify recurring operational problems that may contribute to delivery inefficiency and service disruption.

In [20]:
# Query 7: Incident frequency and severity analysis
incident_analysis <- sqldf("
  SELECT
    incident_type,
    COUNT(*) AS total_incidents,
    SUM(CASE WHEN severity IN ('High', 'Critical') THEN 1 ELSE 0 END) AS high_or_critical_incidents,
    ROUND(AVG(resolved_hours), 2) AS average_resolution_hours
  FROM incidents
  GROUP BY incident_type
  ORDER BY total_incidents DESC, high_or_critical_incidents DESC
")

incident_analysis

incident_type,total_incidents,high_or_critical_incidents,average_resolution_hours
<chr>,<int>,<int>,<dbl>
ProofMissing,46,18,10.77
CustomerNoShow,44,16,13.89
RouteDeviation,43,16,13.73
VehicleFault,37,13,9.15
BatteryAlert,36,10,11.71
AppSyncError,31,7,12.66
TemperatureIssue,29,10,12.92
SafetyNearMiss,14,5,9.67


### 4.8 Query 8: Customer Segmentation and Complaint Behaviour

A SQL join query is used to compare complaint behaviour across customer segments. This helps determine whether particular customer groups generate more complaints and whether the financial impact of complaints differs by customer type.

In [21]:
# Query 8: Customer segmentation and complaint behaviour
customer_complaint_analysis <- sqldf("
  SELECT
    cu.customer_type,
    COUNT(DISTINCT cu.customer_id) AS customers_with_complaints,
    COUNT(c.complaint_id) AS total_complaints,
    ROUND(AVG(c.compensation_amount), 2) AS average_compensation,
    ROUND(AVG(c.resolution_days), 2) AS average_resolution_days
  FROM customers cu
  JOIN complaints c ON cu.customer_id = c.customer_id
  GROUP BY cu.customer_type
  ORDER BY total_complaints DESC
")

customer_complaint_analysis

customer_type,customers_with_complaints,total_complaints,average_compensation,average_resolution_days
<chr>,<int>,<int>,<dbl>,<dbl>
Consumer,172,242,20.62,7.86
SME,40,50,18.47,8.02
Enterprise,21,28,20.23,8.39


### 4.9 Query 9: SQL Query Optimisation Example

An optimised SQL query is used to retrieve completed delivery performance by hub. The query applies filtering before aggregation, selects only the required columns, and groups the data efficiently to reduce unnecessary processing while producing useful operational output.

In [23]:
# Query 9: Optimised SQL query example
optimised_sql_query <- sqldf("
  SELECT
    h.hub_id,
    h.hub_name,
    COUNT(d.delivery_id) AS on_time_deliveries,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS average_operational_cost
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  WHERE d.delivery_status = 'OnTime'
  GROUP BY h.hub_id, h.hub_name
  ORDER BY on_time_deliveries DESC
")

optimised_sql_query

hub_id,hub_name,on_time_deliveries,average_operational_cost
<chr>,<chr>,<int>,<dbl>
H01,North Exchange,93,12.86
H03,East Dock,85,12.59
H04,West Gate,83,12.79
H08,Midtown Relay,80,11.40
H07,Riverside Hub,76,12.79
H02,South Link,70,12.45
H05,Central Core,67,13.59
H06,Airport Hub,62,13.14


### 4.10 Query 10: SQL Explain Plan Before and After Indexing

To demonstrate query optimisation more clearly, the deliveries dataset is written into an in-memory SQLite database through R. An `EXPLAIN QUERY PLAN` statement is used before and after creating an index on `delivery_status`. This shows how indexing can change query execution from a full table scan to indexed retrieval.

In [27]:
# Recreate the in-memory SQLite table and create the index clearly
con <- dbConnect(SQLite(), ":memory:")
dbWriteTable(con, "deliveries_sql", deliveries, overwrite = TRUE)

# Create index on delivery_status
dbExecute(con, "
  CREATE INDEX idx_delivery_status
  ON deliveries_sql(delivery_status);
")

# Verify that the index now exists
dbGetQuery(con, "PRAGMA index_list('deliveries_sql');")

[1] 0

seq,name,unique,origin,partial
<int>,<chr>,<int>,<chr>,<int>
0,idx_delivery_status,0,c,0


In [28]:
# Query execution plan after indexing
after_index_plan <- dbGetQuery(con, "
  EXPLAIN QUERY PLAN
  SELECT delivery_id, delivery_status
  FROM deliveries_sql
  WHERE delivery_status = 'Delayed';
")

after_index_plan

id,parent,notused,detail
<int>,<int>,<int>,<chr>
3,0,61,SEARCH deliveries_sql USING INDEX idx_delivery_status (delivery_status=?)


## 5. Summary of SQL Analysis in R

The SQL analysis in R successfully demonstrated relational data handling, CRUD operations, joins, filtering, grouping, aggregation, and query optimisation using the structured NorthStar datasets.

The analysis identified several important operational issues:

- 35.15% of deliveries were either delayed or failed.
- Hub H05 had the highest non-on-time delivery percentage at 41.74%, followed by H06 at 40.38% and H08 at 37.50%.
- Drivers such as D141 and D165 recorded the lowest average customer ratings among drivers with at least five deliveries.
- High route override deliveries had longer average distances, higher operating costs, lower customer ratings, and a higher non-on-time delivery percentage than deliveries with no route overrides.
- Hub H05 also had the highest average operational cost, making it a priority location for further investigation.
- Failed deliveries generated the highest complaint rate, highest average compensation, and longest complaint resolution time.
- ProofMissing, CustomerNoShow, and RouteDeviation were the most frequent incident types.
- Consumer customers generated the highest complaint volume.

The optimisation example demonstrated that creating an index on `delivery_status` changed the SQLite query plan from a full table scan to indexed retrieval. Overall, the SQL-in-R workflow provided clear evidence of operational inefficiencies and showed how relational querying can support data-driven decision-making within NorthStar.